# 🌐 Simulasi Optimasi Jaringan SDN dengan NetworkX (versi Google Colab)

Notebook ini adalah **alternatif tanpa Ryu/Mininet/WSL**: jaringan disimulasikan sepenuhnya dengan **NetworkX** sehingga bisa langsung dijalankan di **Google Colab** (atau Jupyter mana pun).

Konsep & model mengikuti proyek **reinforcement_and_federated_network** (FedNet Edu):

| Bagian | Isi |
|---|---|
| 1. Topologi | node acak pada area, link ke *k* tetangga terdekat, delay dari jarak, node dibagi ke **region** (k-means) |
| 2. Kondisi trafik | utilisasi link mengikuti profil region (intensitas, jam puncak) → delay **antrean M/M/1** |
| 3. Routing klasik | Hop-Count, Dijkstra (delay dasar), Dijkstra (delay aktual = *oracle*), Widest-Path, QoS |
| 4. Reinforcement Learning | agen Q-learning dengan **reward terurai** (+tiba, ±kemajuan, −delay, −hop, −loop, −kongesti, −timeout) |
| 5. Federated Q-Routing | tiap region = klien federated; tabel-Q digabung tiap ronde → dibandingkan dengan lokal, terpusat, oracle |
| 6. Traffic engineering | beberapa flow besar → jalur terpendek vs **load-aware** (max-min fair) |

### Cara pakai
1. **Runtime → Run all** (Ctrl+F9). Tidak perlu instalasi apa pun.
2. Ubah parameter lewat **form** di sisi kanan tiap sel (slider/dropdown), lalu jalankan ulang sel tersebut dan sel-sel sesudahnya.
3. Semua hasil (CSV & PNG) tersimpan di folder `hasil_networkx/` dan bisa diunduh sebagai ZIP di bagian akhir.

In [ ]:
#@title 📦 Import library (sudah tersedia di Colab)
import math, random, time, json, itertools, os, shutil
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
try:
    import ipywidgets as widgets
    from ipywidgets import interact
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

OUT = Path("hasil_networkx/01_optimasi")
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
REGION_COLORS = ["#e8743b", "#1baf7a", "#a259d9", "#c98500", "#19a3b5", "#e34948", "#7a8c00", "#8a6d3b"]
print(f"networkx {nx.__version__} | numpy {np.__version__} | pandas {pd.__version__} | Colab: {IN_COLAB} | ipywidgets: {HAS_WIDGETS}")

## 1. Topologi jaringan
**Parameter:**
- `JENIS_TOPOLOGI` – *acak* (seperti generator FedNet Edu), *contoh 10 switch* (topologi proyek software_defined_network), *grid*, atau *ring + chord*.
- `JUMLAH_NODE`, `LUAS_WILAYAH_M` – jumlah switch dan ukuran area (meter). Delay dasar link = 0.5–15 ms sebanding jarak.
- `TETANGGA_TERDEKAT` – tiap node dihubungkan ke *k* tetangga terdekat (makin besar → makin banyak jalur alternatif).
- `JUMLAH_REGION` – banyaknya region/domain (nanti menjadi klien federated).

In [ ]:
#@title ⚙️ Parameter topologi
JENIS_TOPOLOGI = "acak (k-tetangga terdekat)"  #@param ["acak (k-tetangga terdekat)", "contoh 10 switch", "grid", "ring + chord"]
JUMLAH_NODE = 20  #@param {type:"slider", min:6, max:60, step:1}
LUAS_WILAYAH_M = 1000  #@param {type:"slider", min:200, max:5000, step:100}
TETANGGA_TERDEKAT = 3  #@param {type:"slider", min:2, max:6, step:1}
JUMLAH_REGION = 4  #@param {type:"slider", min:1, max:8, step:1}
SEED = 42  #@param {type:"integer"}

In [ ]:
#@title 🏗️ Bangun topologi
CONTOH_10_SWITCH = [  # (u, v, delay_ms, bandwidth_mbps, loss) — data/topology.json proyek software_defined_network
    ("N1", "N2", 2, 100, .001), ("N1", "N3", 5, 60, .003), ("N2", "N3", 1, 100, .001), ("N2", "N4", 4, 80, .002),
    ("N3", "N4", 1, 100, .001), ("N3", "N5", 6, 50, .004), ("N4", "N5", 2, 90, .002), ("N4", "N6", 3, 70, .002),
    ("N5", "N6", 1, 100, .001), ("N5", "N7", 4, 60, .003), ("N6", "N7", 2, 90, .002), ("N6", "N8", 5, 50, .004),
    ("N7", "N8", 1, 100, .001), ("N7", "N9", 3, 80, .002), ("N8", "N9", 2, 90, .002), ("N8", "N10", 4, 60, .003),
    ("N9", "N10", 1, 100, .001), ("N2", "N5", 7, 40, .005), ("N3", "N6", 6, 45, .004), ("N1", "N4", 8, 35, .005)]

def kmeans(points, k, seed=0, iters=50):
    """k-means sederhana (numpy) untuk membagi node ke region berdasarkan posisi."""
    pts = np.asarray(points, float)
    k = max(1, min(k, len(pts)))
    rng = np.random.default_rng(seed)
    cent = pts[rng.choice(len(pts), k, replace=False)]
    for _ in range(iters):
        lab = np.argmin(((pts[:, None] - cent[None]) ** 2).sum(-1), axis=1)
        new = np.array([pts[lab == j].mean(0) if (lab == j).any() else cent[j] for j in range(k)])
        if np.allclose(new, cent):
            break
        cent = new
    return lab

def build_topology(kind, n, area, k, n_regions, seed):
    rng = np.random.default_rng(seed)
    G = nx.Graph()
    if kind == "contoh 10 switch":
        G.add_edges_from((u, v, dict(base_delay=float(d), bandwidth=float(b), loss=l)) for u, v, d, b, l in CONTOH_10_SWITCH)
        pos = nx.kamada_kawai_layout(G)
        pos = {u: ((x + 1) / 2 * area, (y + 1) / 2 * area) for u, (x, y) in pos.items()}
    else:
        if kind == "grid":
            side = max(2, round(math.sqrt(n)))
            H = nx.grid_2d_graph(side, math.ceil(n / side))
            H = H.subgraph(list(H.nodes())[:n]).copy()
            step = area / (side + 1)
            pos = {f"N{i + 1}": ((c[0] + 1) * step, (c[1] + 1) * step) for i, c in enumerate(H.nodes())}
            mapping = {c: f"N{i + 1}" for i, c in enumerate(H.nodes())}
            G.add_edges_from((mapping[a], mapping[b]) for a, b in H.edges())
        elif kind == "ring + chord":
            pos = {f"N{i + 1}": (area / 2 * (1 + 0.9 * math.cos(2 * math.pi * i / n)),
                                 area / 2 * (1 + 0.9 * math.sin(2 * math.pi * i / n))) for i in range(n)}
            for i in range(n):
                G.add_edge(f"N{i + 1}", f"N{(i + 1) % n + 1}")
            for _ in range(max(1, n // 3)):
                a, b = rng.choice(n, 2, replace=False)
                G.add_edge(f"N{a + 1}", f"N{b + 1}")
        else:  # acak: k tetangga terdekat (gaya generate_topology FedNet Edu)
            pos = {f"N{i + 1}": (float(rng.uniform(0, area)), float(rng.uniform(0, area))) for i in range(n)}
            G.add_nodes_from(pos)
            for u in pos:
                near = sorted((v for v in pos if v != u), key=lambda v: math.dist(pos[u], pos[v]))
                for v in near[:k]:
                    G.add_edge(u, v)
        while not nx.is_connected(G):  # jamin terhubung
            comps = list(nx.connected_components(G))
            a, b = min(((x, y) for x in comps[0] for y in set().union(*comps[1:])), key=lambda p: math.dist(pos[p[0]], pos[p[1]]))
            G.add_edge(a, b)
        dmax = math.hypot(area, area)
        for u, v, d in G.edges(data=True):
            d["base_delay"] = round(0.5 + 14.5 * math.dist(pos[u], pos[v]) / dmax, 3)
            d["bandwidth"] = float(rng.choice([10, 20, 50, 100]))
            d["loss"] = round(float(rng.uniform(0.0005, 0.01)), 4)
    for u, v, d in G.edges(data=True):
        d["distance_m"] = round(math.dist(pos[u], pos[v]), 1)
        d["traffic_weight"] = round(float(rng.uniform(0.6, 1.4)), 2)
    nodes = sorted(G.nodes(), key=lambda s: int(s[1:]))
    labels = kmeans([pos[u] for u in nodes], n_regions, seed=seed)
    for u, lab in zip(nodes, labels):
        G.nodes[u]["pos"] = pos[u]
        G.nodes[u]["region"] = f"R{lab + 1}"
    return G

G = build_topology(JENIS_TOPOLOGI, JUMLAH_NODE, LUAS_WILAYAH_M, TETANGGA_TERDEKAT, JUMLAH_REGION, SEED)
NODES = sorted(G.nodes(), key=lambda s: int(s[1:]))
POS = nx.get_node_attributes(G, "pos")
REGIONS = sorted({G.nodes[u]["region"] for u in G}, key=lambda r: int(r[1:]))
RCOLOR = {r: REGION_COLORS[i % len(REGION_COLORS)] for i, r in enumerate(REGIONS)}

def draw_base(ax, edge_color="#c9d1d9", width=1.2, node_size=260, labels=True):
    nx.draw_networkx_edges(G, POS, ax=ax, edge_color=edge_color, width=width)
    nx.draw_networkx_nodes(G, POS, ax=ax, node_size=node_size, node_color=[RCOLOR[G.nodes[u]["region"]] for u in G],
                           edgecolors="white", linewidths=1.2)
    if labels:
        nx.draw_networkx_labels(G, POS, ax=ax, font_size=7, font_color="white", font_weight="bold")
    ax.set_aspect("equal"); ax.axis("off")

fig, ax = plt.subplots(figsize=(8, 6))
bw = np.array([G[u][v]["bandwidth"] for u, v in G.edges()])
draw_base(ax, width=0.6 + bw / 30, edge_color="#9aa5b1")
ax.legend(handles=[Line2D([], [], marker="o", ls="", color=RCOLOR[r], label=f"Region {r}") for r in REGIONS] +
          [Line2D([], [], color="#9aa5b1", lw=0.6 + b / 30, label=f"{b:g} Mbps") for b in sorted(set(bw))],
          loc="upper left", bbox_to_anchor=(1, 1), fontsize=8)
ax.set_title(f"Topologi: {G.number_of_nodes()} node, {G.number_of_edges()} link (warna = region, tebal = bandwidth)")
plt.tight_layout(); plt.savefig(OUT / "1_topologi.png", dpi=150); plt.show()

links_df = pd.DataFrame([dict(link=f"{u}-{v}", **d) for u, v, d in G.edges(data=True)])
display(links_df.head(8))

## 2. Kondisi trafik saat simulasi
Tiap region punya **profil trafik** (intensitas & jam puncak berbeda → kondisi non-IID seperti FedNet Edu). Utilisasi link ρ mengikuti region pemiliknya dan pola harian; delay aktual memakai model antrean **M/M/1**:

$$\text{delay} = \text{delay}_{dasar} + 8\cdot\frac{12}{\text{bandwidth}}\cdot\frac{\rho}{1-\rho} + \text{loss}\cdot 200\,\text{ms}$$

Saat ρ mendekati 1, delay melonjak tajam — inilah alasan routing perlu **sadar beban** (load-aware).

**Parameter:** `JAM_SIMULASI` (0–23), `SKALA_TRAFIK` (pengali beban seluruh jaringan; >1 = jam sibuk ekstrem).

In [ ]:
#@title ⚙️ Parameter trafik
JAM_SIMULASI = 19  #@param {type:"slider", min:0, max:23, step:1}
SKALA_TRAFIK = 1.0  #@param {type:"slider", min:0.2, max:1.6, step:0.1}

In [ ]:
#@title 🚦 Hitung utilisasi & delay aktual tiap link
rng_t = np.random.default_rng(SEED + 1)
intens = np.linspace(0.25, 0.85, len(REGIONS)) if len(REGIONS) > 1 else np.array([0.6])
rng_t.shuffle(intens)
PROFIL = pd.DataFrame({"region": REGIONS, "intensitas": np.round(intens, 2),
                       "jam_puncak": rng_t.integers(8, 22, len(REGIONS)),
                       "burstiness": np.round(rng_t.uniform(0.03, 0.12, len(REGIONS)), 3)}).set_index("region")

def diurnal(hour, peak):
    return 0.65 + 0.35 * np.sin(2 * np.pi * (hour - peak + 6) / 24)

def true_delay(rho, bw, base, loss):
    rho = np.clip(rho, 0.01, 0.97)
    return base + 8.0 * (12.0 / bw) * rho / (1 - rho) + loss * 200.0

def apply_traffic(G, hour, scale, seed):
    r = np.random.default_rng(seed)
    for u, v, d in G.edges(data=True):
        owner = G.nodes[u]["region"]                 # link dikelola region node pertama
        p = PROFIL.loc[owner]
        rho = p.intensitas * d["traffic_weight"] * diurnal(hour, p.jam_puncak) * scale + r.normal(0, p.burstiness * 0.5)
        d["owner"] = owner
        d["util"] = float(np.clip(rho, 0.01, 0.97))
        d["delay"] = float(true_delay(d["util"], d["bandwidth"], d["base_delay"], d["loss"]))

apply_traffic(G, JAM_SIMULASI, SKALA_TRAFIK, SEED + 2)
display(PROFIL)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
util = np.array([G[u][v]["util"] for u, v in G.edges()])
ax = axes[0]
nx.draw_networkx_edges(G, POS, ax=ax, edge_color=util, edge_cmap=plt.cm.RdYlGn_r, edge_vmin=0, edge_vmax=1, width=1 + 5 * util)
draw_base(ax, width=0)
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdYlGn_r, norm=plt.Normalize(0, 1)); plt.colorbar(sm, ax=ax, fraction=0.04, label="utilisasi ρ")
ax.set_title(f"Utilisasi link pada jam {JAM_SIMULASI}:00")
ax = axes[1]
rho = np.linspace(0.01, 0.97, 200)
for b in sorted({G[u][v]["bandwidth"] for u, v in G.edges()}):
    ax.plot(rho, true_delay(rho, b, 2.0, 0.003), label=f"{b:g} Mbps")
ax.scatter(util, [G[u][v]["delay"] - G[u][v]["base_delay"] + 2 for u, v in G.edges()], s=14, color="black", zorder=3, label="link saat ini")
ax.set_xlabel("utilisasi ρ"); ax.set_ylabel("delay (ms), delay dasar 2 ms"); ax.set_yscale("log"); ax.legend(fontsize=8)
ax.set_title("Model antrean M/M/1: delay melonjak saat ρ → 1")
plt.tight_layout(); plt.savefig(OUT / "2_trafik.png", dpi=150); plt.show()

## 3. Routing klasik
Pilih pasangan **sumber → tujuan** (isi `"auto"` untuk memakai pasangan terjauh). Algoritma yang dibandingkan:

| Algoritma | Bobot | Keterangan |
|---|---|---|
| Hop-Count | jumlah hop | perilaku routing default (tanpa optimasi) |
| Dijkstra (delay dasar) | delay propagasi saja | rute statis, **tidak tahu beban** |
| Dijkstra (delay aktual) | delay M/M/1 saat ini | **oracle**: controller SDN dengan statistik real-time |
| Widest-Path | maksimalkan bandwidth bottleneck | tie-break delay aktual |
| QoS | kombinasi delay aktual, utilisasi, loss | bobot diatur di form |

In [ ]:
#@title ⚙️ Parameter routing
SUMBER = "auto"  #@param {type:"string"}
TUJUAN = "auto"  #@param {type:"string"}
BOBOT_QOS_DELAY = 0.5  #@param {type:"slider", min:0, max:1, step:0.05}
BOBOT_QOS_UTILISASI = 0.3  #@param {type:"slider", min:0, max:1, step:0.05}
BOBOT_QOS_LOSS = 0.2  #@param {type:"slider", min:0, max:1, step:0.05}

In [ ]:
#@title 🧭 Jalankan & bandingkan algoritma routing
def path_edges(p):
    return list(zip(p[:-1], p[1:]))

def path_metrics(p):
    e = path_edges(p)
    return dict(hop=len(e), delay_aktual_ms=sum(G[u][v]["delay"] for u, v in e),
                delay_dasar_ms=sum(G[u][v]["base_delay"] for u, v in e),
                bottleneck_mbps=min(G[u][v]["bandwidth"] for u, v in e),
                utilisasi_maks=max(G[u][v]["util"] for u, v in e),
                loss_pct=100 * (1 - np.prod([1 - G[u][v]["loss"] for u, v in e])))

dmax = max(d["delay"] for *_, d in G.edges(data=True)); lmax = max(d["loss"] for *_, d in G.edges(data=True))
for u, v, d in G.edges(data=True):
    d["qos"] = BOBOT_QOS_DELAY * d["delay"] / dmax + BOBOT_QOS_UTILISASI * d["util"] + BOBOT_QOS_LOSS * d["loss"] / lmax + 1e-6

def widest(s, t):
    for thr in sorted({d["bandwidth"] for *_, d in G.edges(data=True)}, reverse=True):
        H = G.edge_subgraph([(u, v) for u, v, d in G.edges(data=True) if d["bandwidth"] >= thr])
        if s in H and t in H and nx.has_path(H, s, t):
            return nx.dijkstra_path(H, s, t, weight="delay")

ROUTERS = {"Hop-Count": lambda s, t: nx.shortest_path(G, s, t),
           "Dijkstra (delay dasar)": lambda s, t: nx.dijkstra_path(G, s, t, weight="base_delay"),
           "Dijkstra (delay aktual)": lambda s, t: nx.dijkstra_path(G, s, t, weight="delay"),
           "Widest-Path": widest,
           "QoS": lambda s, t: nx.dijkstra_path(G, s, t, weight="qos")}
ALG_COLORS = dict(zip(ROUTERS, ["#9aa5b1", "#2f6db5", "#d6006f", "#1baf7a", "#e8743b"]))

hop_all = dict(nx.all_pairs_shortest_path_length(G))
if SUMBER in G and TUJUAN in G and SUMBER != TUJUAN:
    SRC, DST = SUMBER, TUJUAN
else:
    SRC, DST = max(((s, t) for s in NODES for t in NODES if s != t),
                   key=lambda p: (hop_all[p[0]][p[1]], math.dist(POS[p[0]], POS[p[1]])))
print(f"Pasangan: {SRC} → {DST}")

rows = []
for name, fn in ROUTERS.items():
    p = fn(SRC, DST)
    rows.append(dict(algoritma=name, jalur="→".join(p), **path_metrics(p)))
res_pair = pd.DataFrame(rows).set_index("algoritma")
oracle_delay = res_pair.loc["Dijkstra (delay aktual)", "delay_aktual_ms"]
res_pair["selisih_vs_oracle_%"] = 100 * (res_pair.delay_aktual_ms / oracle_delay - 1)
display(res_pair.round(2))

fig, axes = plt.subplots(1, len(ROUTERS), figsize=(4.2 * len(ROUTERS), 4))
for ax, (name, fn) in zip(axes, ROUTERS.items()):
    p = fn(SRC, DST)
    draw_base(ax, node_size=150, labels=False)
    nx.draw_networkx_edges(G, POS, ax=ax, edgelist=path_edges(p), edge_color=ALG_COLORS[name], width=4)
    nx.draw_networkx_nodes(G, POS, ax=ax, nodelist=[SRC, DST], node_color="black", node_size=200)
    m = res_pair.loc[name]
    ax.set_title(f"{name}\n{m.hop:.0f} hop · {m.delay_aktual_ms:.1f} ms · bott {m.bottleneck_mbps:g} M", fontsize=9)
plt.suptitle(f"Jalur {SRC} → {DST} (hitam = sumber/tujuan)")
plt.tight_layout(); plt.savefig(OUT / "3_jalur_routing.png", dpi=150); plt.show()

# evaluasi semua pasangan: rata-rata selisih terhadap oracle
pairs = [(s, t) for s in NODES for t in NODES if s != t]
oracle_len = {s: nx.single_source_dijkstra_path_length(G, s, weight="delay") for s in NODES}
summ = []
for name, fn in ROUTERS.items():
    gaps, hops = [], []
    for s, t in pairs:
        m = path_metrics(fn(s, t))
        gaps.append(100 * (m["delay_aktual_ms"] / oracle_len[s][t] - 1)); hops.append(m["hop"])
    summ.append(dict(algoritma=name, rata_selisih_vs_oracle_pct=np.mean(gaps),
                     pct_jalur_optimal=100 * np.mean(np.array(gaps) < 0.5), rata_hop=np.mean(hops)))
summ = pd.DataFrame(summ).set_index("algoritma")
summ.to_csv(OUT / "3_ringkasan_semua_pasangan.csv")
print(f"Evaluasi semua {len(pairs)} pasangan (delay aktual):")
display(summ.round(2))

## 4. Reinforcement Learning: reward plus/minus per langkah
Satu agen Q-learning belajar merutekan paket dari **sumber ke tujuan** (pasangan di atas). Setiap langkah `x → y` menerima reward yang **terurai** menjadi komponen (sama dengan `rl_reward.py` FedNet Edu):

| Komponen | Tanda | Rumus |
|---|---|---|
| Tiba di target | + | `REWARD_TIBA` bila y = tujuan |
| Kemajuan | ± | `BOBOT_KEMAJUAN × (hop_ke_tujuan(x) − hop_ke_tujuan(y))` |
| Delay link | − | `BOBOT_DELAY × delay aktual` |
| Biaya hop | − | `PENALTI_HOP` tiap langkah |
| Loop | − | `PENALTI_LOOP` bila kembali ke node yang sudah dikunjungi |
| Kongesti | − | `PENALTI_KONGESTI × (ρ − ambang)/(1 − ambang)` bila ρ > `AMBANG_KONGESTI` |
| Timeout | − | `PENALTI_TIMEOUT` bila batas hop habis |

Update: $Q(s,a) \leftarrow Q(s,a) + \alpha\,[\,r + \gamma \max_{a'} Q(s',a') - Q(s,a)\,]$, eksplorasi ε-greedy yang meluruh.

In [ ]:
#@title ⚙️ Parameter agen RL
EPISODE = 300  #@param {type:"slider", min:20, max:2000, step:20}
ALPHA = 0.3  #@param {type:"slider", min:0.01, max:1, step:0.01}
GAMMA = 0.95  #@param {type:"slider", min:0.5, max:1, step:0.01}
EPSILON_AWAL = 0.5  #@param {type:"slider", min:0, max:1, step:0.05}
PELURUHAN_EPSILON = 0.985  #@param {type:"slider", min:0.9, max:1, step:0.001}
REWARD_TIBA = 100  #@param {type:"number"}
BOBOT_KEMAJUAN = 5  #@param {type:"number"}
BOBOT_DELAY = 1.0  #@param {type:"number"}
PENALTI_HOP = 1  #@param {type:"number"}
PENALTI_LOOP = 20  #@param {type:"number"}
AMBANG_KONGESTI = 0.8  #@param {type:"slider", min:0.3, max:0.97, step:0.01}
PENALTI_KONGESTI = 30  #@param {type:"number"}
PENALTI_TIMEOUT = 50  #@param {type:"number"}

In [ ]:
#@title 🤖 Latih agen RL
COMP = ["tiba", "kemajuan", "delay", "hop", "loop", "kongesti", "timeout"]
COMP_COLOR = dict(zip(COMP, ["#1baf7a", "#19a3b5", "#2f6db5", "#9aa5b1", "#e34948", "#e8743b", "#7a1f1f"]))

def step_reward(x, y, visited, hopd, last):
    d = G[x][y]
    thr = AMBANG_KONGESTI
    return {"tiba": REWARD_TIBA if y == DST else 0.0,
            "kemajuan": BOBOT_KEMAJUAN * (hopd[x] - hopd[y]),
            "delay": -BOBOT_DELAY * d["delay"],
            "hop": -PENALTI_HOP,
            "loop": -PENALTI_LOOP if y in visited else 0.0,
            "kongesti": -PENALTI_KONGESTI * (d["util"] - thr) / (1 - thr) if d["util"] > thr else 0.0,
            "timeout": -PENALTI_TIMEOUT if (last and y != DST) else 0.0}

def train_rl(seed=SEED):
    r = np.random.default_rng(seed)
    hopd = nx.single_source_shortest_path_length(G, DST)
    nb = {u: sorted(G.neighbors(u), key=lambda s: int(s[1:])) for u in G}
    Q = {u: np.zeros(len(nb[u])) for u in G}
    max_hops = max(3, int(1.5 * len(G)))
    eps, log = EPSILON_AWAL, []
    for ep in range(1, EPISODE + 1):
        x, visited, path, sums = SRC, {SRC}, [SRC], dict.fromkeys(COMP, 0.0)
        explore = 0
        for k in range(1, max_hops + 1):
            if r.random() < eps:
                i = int(r.integers(len(nb[x]))); explore += 1
            else:
                i = int(np.argmax(Q[x]))
            y = nb[x][i]
            comp = step_reward(x, y, visited, hopd, k == max_hops)
            rew = sum(comp.values())
            future = 0.0 if y == DST else Q[y].max()
            Q[x][i] += ALPHA * (rew + GAMMA * future - Q[x][i])
            for c in COMP:
                sums[c] += comp[c]
            visited.add(y); path.append(y); x = y
            if x == DST:
                break
        total = sum(sums.values())
        log.append(dict(episode=ep, total=total, plus=sum(v for v in sums.values() if v > 0),
                        minus=sum(v for v in sums.values() if v < 0), tiba=x == DST, langkah=len(path) - 1,
                        eksplorasi=explore, epsilon=eps, jalur=path, **{f"r_{c}": sums[c] for c in COMP}))
        eps = max(0.02, eps * PELURUHAN_EPSILON)
    return Q, nb, pd.DataFrame(log), hopd

Q_rl, NB, rl_log, HOPD = train_rl()

# kebijakan akhir (greedy) + rincian reward per langkah
x, visited, steps = SRC, {SRC}, []
for k in range(1, max(3, int(1.5 * len(G))) + 1):
    i = int(np.argmax(Q_rl[x])); y = NB[x][i]
    comp = step_reward(x, y, visited, HOPD, False)
    steps.append(dict(langkah=k, dari=x, ke=y, Q=Q_rl[x][i], **comp, total=sum(comp.values())))
    if y in visited or y == DST:
        visited.add(y); x = y; break
    visited.add(y); x = y
policy_path = [SRC] + [s["ke"] for s in steps]
ok = policy_path[-1] == DST and len(set(policy_path)) == len(policy_path)
oracle_path = nx.dijkstra_path(G, SRC, DST, weight="delay")
first_ok = rl_log[rl_log.tiba].episode.min() if rl_log.tiba.any() else None
print(f"Episode pertama tiba di tujuan : {first_ok}")
print(f"% episode tiba (100 terakhir)  : {100 * rl_log.tiba.tail(100).mean():.0f}%")
print(f"Jalur kebijakan akhir          : {' → '.join(policy_path)}  {'✔' if ok else '✘ belum konvergen'}")
if ok:
    print(f"Delay aktual RL vs oracle      : {path_metrics(policy_path)['delay_aktual_ms']:.2f} ms vs {path_metrics(oracle_path)['delay_aktual_ms']:.2f} ms ({' → '.join(oracle_path)})")
display(pd.DataFrame(steps).round(2))
rl_log.drop(columns="jalur").to_csv(OUT / "4_rl_episode.csv", index=False)

In [ ]:
#@title 📈 Visualisasi pembelajaran RL
win = max(1, EPISODE // 30)
binned = rl_log.groupby((rl_log.episode - 1) // win)[[f"r_{c}" for c in COMP] + ["total"]].mean()
xs = binned.index * win + win / 2
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={"width_ratios": [2, 1]})
ax = axes[0]
pos_base, neg_base = np.zeros(len(binned)), np.zeros(len(binned))
for c in COMP:
    v = binned[f"r_{c}"].values
    ax.bar(xs, np.clip(v, 0, None), win * 0.9, bottom=pos_base, color=COMP_COLOR[c], label=c)
    ax.bar(xs, np.clip(v, None, 0), win * 0.9, bottom=neg_base, color=COMP_COLOR[c])
    pos_base += np.clip(v, 0, None); neg_base += np.clip(v, None, 0)
ax.plot(xs, binned.total, color="black", lw=2, label="total return")
ax.axhline(0, color="black", lw=0.6)
ax.set_xlabel("episode"); ax.set_ylabel(f"reward rata-rata per {win} episode")
ax.set_title("Komponen reward per episode (atas = hadiah, bawah = hukuman)"); ax.legend(fontsize=8, ncol=4)
ax = axes[1]
draw_base(ax, node_size=150, labels=False)
nx.draw_networkx_edges(G, POS, ax=ax, edgelist=path_edges(oracle_path), edge_color="#d6006f", width=7, alpha=0.35)
if ok:
    nx.draw_networkx_edges(G, POS, ax=ax, edgelist=path_edges(policy_path), edge_color="black", width=2.5, style="dashed")
nx.draw_networkx_nodes(G, POS, ax=ax, nodelist=[SRC, DST], node_color="black", node_size=200)
ax.legend(handles=[Line2D([], [], color="#d6006f", lw=6, alpha=0.35, label="oracle (delay aktual)"),
                   Line2D([], [], color="black", lw=2.5, ls="--", label="kebijakan RL")], fontsize=8, loc="lower left")
ax.set_title(f"Jalur akhir {SRC} → {DST}")
plt.tight_layout(); plt.savefig(OUT / "4_rl_pembelajaran.png", dpi=150); plt.show()

In [ ]:
#@title 🎞️ Putar ulang jalur yang ditempuh agen per episode (geser slider)
def show_episode(episode=1):
    row = rl_log.iloc[episode - 1]
    fig, ax = plt.subplots(figsize=(6.5, 5))
    draw_base(ax, node_size=170, labels=True)
    e = path_edges(row.jalur)
    nx.draw_networkx_edges(G, POS, ax=ax, edgelist=e, edge_color="#e8743b", width=3, arrows=True,
                           arrowstyle="-|>", arrowsize=14, connectionstyle="arc3,rad=0.12")
    nx.draw_networkx_nodes(G, POS, ax=ax, nodelist=[SRC, DST], node_color="black", node_size=220)
    ax.set_title(f"Episode {episode}: {'TIBA' if row.tiba else 'gagal'} · {row.langkah} langkah · return {row.total:.1f} · ε={row.epsilon:.2f}", fontsize=10)
    plt.show()

if HAS_WIDGETS:
    interact(show_episode, episode=widgets.IntSlider(min=1, max=EPISODE, step=1, value=1, continuous_update=False))
else:
    for e in sorted({1, max(1, EPISODE // 10), EPISODE // 2, EPISODE}):
        show_episode(e)

## 5. Federated Q-Routing
Setiap **region = klien federated** (mis. controller SDN lokal/ISP regional) yang menjalankan agen **Q-Routing** (Boyan & Littman):

$$Q_x(d,y) \leftarrow Q_x(d,y) + \alpha\,[\,c(x,y) + \min_z Q_y(d,z) - Q_x(d,y)\,]$$

$Q_x(d,y)$ = estimasi delay total bila paket di node *x* menuju *d* diteruskan ke tetangga *y*. Klien hanya belajar dari paket yang **berasal dari region-nya** (tujuan sebagian besar lokal: `LOKALITAS_TUJUAN`). Tiap ronde server **menggabungkan tabel-Q** lalu membagikan tabel global — pengetahuan rute dibagi tanpa membagikan data trafik mentah.

Pembanding: **Global (federated)**, **Lokal saja** (tiap klien belajar sendiri), **Terpusat** (satu agen melihat semua paket = batas atas), **Oracle** (Dijkstra delay aktual).

In [ ]:
#@title ⚙️ Parameter federated Q-routing
RONDE = 20  #@param {type:"slider", min:1, max:60, step:1}
EPISODE_PER_RONDE = 200  #@param {type:"slider", min:20, max:1000, step:20}
ALPHA_Q = 0.5  #@param {type:"slider", min:0.05, max:1, step:0.05}
EPSILON_Q = 0.3  #@param {type:"slider", min:0, max:1, step:0.05}
LOKALITAS_TUJUAN = 0.6  #@param {type:"slider", min:0, max:1, step:0.05}
METODE_AGREGASI = "visit_weighted"  #@param ["visit_weighted", "fedavg"]
JUMLAH_PASANGAN_EVALUASI = 60  #@param {type:"slider", min:10, max:300, step:10}

In [ ]:
#@title 🤝 Latih federated Q-routing
IDX = {u: i for i, u in enumerate(NODES)}
N = len(NODES)
NBI = [[IDX[v] for v in sorted(G.neighbors(u), key=lambda s: int(s[1:]))] for u in NODES]
COST = {(IDX[u], IDX[v]): G[u][v]["delay"] for u, v in G.edges()}
COST.update({(b, a): c for (a, b), c in list(COST.items())})
REG_NODES = {r: [IDX[u] for u in NODES if G.nodes[u]["region"] == r] for r in REGIONS}
MAX_HOPS = 2 * N

def new_q():
    return [np.zeros((N, len(NBI[x]))) for x in range(N)]

def run_episodes(Q, V, sources, local, episodes, eps, r):
    for _ in range(episodes):
        s = sources[r.integers(len(sources))]
        pool = local if (len(local) > 1 and r.random() < LOKALITAS_TUJUAN) else range(N)
        pool = [p for p in pool if p != s]
        d = pool[r.integers(len(pool))]
        x = s
        for _h in range(MAX_HOPS):
            q = Q[x][d]
            k = int(r.integers(len(q))) if r.random() < eps else int(np.argmin(q))
            y = NBI[x][k]
            c = COST[(x, y)] * float(np.exp(r.normal(0, 0.05)))
            q[k] += ALPHA_Q * (c + (0.0 if y == d else Q[y][d].min()) - q[k])
            V[x][d][k] += 1
            x = y
            if x == d:
                break

def aggregate(Qs, Vs, prev):
    out = []
    for x in range(N):
        if METODE_AGREGASI == "fedavg":
            out.append(np.mean([Q[x] for Q in Qs], axis=0))
        else:
            w = sum(V[x] for V in Vs)
            num = sum(Q[x] * V[x] for Q, V in zip(Qs, Vs))
            out.append(np.where(w > 0, num / np.maximum(w, 1e-12), prev[x]))
    return out

rng_e = np.random.default_rng(SEED + 5)
EVAL = [pairs[i] for i in rng_e.choice(len(pairs), min(JUMLAH_PASANGAN_EVALUASI, len(pairs)), replace=False)]

def greedy_path(Q, V, s, t):
    x, path = IDX[s], [s]
    for _ in range(MAX_HOPS):
        if x == IDX[t]:
            return path
        q, v = Q[x][IDX[t]], V[x][IDX[t]]
        cand = np.where(v > 0)[0] if (v > 0).any() else np.arange(len(q))
        x = NBI[x][cand[np.argmin(q[cand])]]
        if NODES[x] in path:
            return None
        path.append(NODES[x])
    return None

def evaluate(Q, V):
    ok, opt, gaps = 0, 0, []
    for s, t in EVAL:
        p = greedy_path(Q, V, s, t)
        if p:
            ok += 1
            g = 100 * (path_metrics(p)["delay_aktual_ms"] / oracle_len[s][t] - 1)
            gaps.append(g); opt += g < 0.5
    n = len(EVAL)
    return dict(jalur_berhasil_pct=100 * ok / n, jalur_optimal_pct=100 * opt / n,
                rata_selisih_pct=np.mean(gaps) if gaps else np.nan)

t0 = time.time()
Qg, round_log, round_paths = new_q(), [], []
Vtot = [np.zeros_like(q) for q in Qg]
Q_local = {r: new_q() for r in REGIONS}; V_local = {r: [np.zeros_like(q) for q in new_q()] for r in REGIONS}
Q_cent, V_cent = new_q(), [np.zeros_like(q) for q in new_q()]
all_src = list(range(N))
for rnd in range(1, RONDE + 1):
    eps = max(0.02, EPSILON_Q * 0.9 ** (rnd - 1))
    Qs, Vs = [], []
    for ci, reg in enumerate(REGIONS):
        r = np.random.default_rng([SEED, rnd, ci])
        Qc = [q.copy() for q in Qg]; Vc = [np.zeros_like(q) for q in Qg]
        run_episodes(Qc, Vc, REG_NODES[reg], REG_NODES[reg], EPISODE_PER_RONDE, eps, r)
        Qs.append(Qc); Vs.append(Vc)
        # lokal saja: klien yang sama, tanpa berbagi
        run_episodes(Q_local[reg], V_local[reg], REG_NODES[reg], REG_NODES[reg], EPISODE_PER_RONDE, eps, np.random.default_rng([SEED, rnd, ci, 7]))
    Qg = aggregate(Qs, Vs, Qg)
    for V in Vs:
        for x in range(N):
            Vtot[x] += V[x]
    run_episodes(Q_cent, V_cent, all_src, all_src, EPISODE_PER_RONDE * len(REGIONS), eps, np.random.default_rng([SEED, rnd, 99]))
    round_log.append(dict(ronde=rnd, model="Global (federated)", **evaluate(Qg, Vtot)))
    round_log.append(dict(ronde=rnd, model="Terpusat", **evaluate(Q_cent, V_cent)))
    for reg in REGIONS:
        round_log.append(dict(ronde=rnd, model=f"Lokal {reg}", **evaluate(Q_local[reg], V_local[reg])))
    round_paths.append(greedy_path(Qg, Vtot, SRC, DST))
round_df = pd.DataFrame(round_log)
round_df.to_csv(OUT / "5_federated_per_ronde.csv", index=False)
final = round_df[round_df.ronde == RONDE].set_index("model").drop(columns="ronde")
final.loc["Oracle"] = [100.0, 100.0, 0.0]
print(f"Selesai {RONDE} ronde × {len(REGIONS)} klien × {EPISODE_PER_RONDE} episode dalam {time.time() - t0:.1f} s")
print(f"Evaluasi pada {len(EVAL)} pasangan acak (sumber & tujuan bisa beda region):")
display(final.round(1))

In [ ]:
#@title 📈 Visualisasi federated Q-routing
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, col, lbl in zip(axes[:2], ["jalur_optimal_pct", "rata_selisih_pct"], ["% jalur optimal ↑", "rata-rata selisih delay vs oracle (%) ↓"]):
    for model, g in round_df.groupby("model"):
        is_local = model.startswith("Lokal")
        color = RCOLOR[model.split()[-1]] if is_local else ("#d6006f" if model.startswith("Global") else "black")
        ax.plot(g.ronde, g[col], color=color, lw=1.2 if is_local else 2.5, ls=":" if is_local else "-", label=model)
    ax.set_xlabel("ronde"); ax.set_title(lbl, fontsize=10)
axes[0].legend(fontsize=7)
ax = axes[2]
models = final.index.tolist()
colors = [RCOLOR.get(m.split()[-1], "#d6006f" if m.startswith("Global") else ("black" if m == "Terpusat" else "#9aa5b1")) for m in models]
ax.barh(models, final.jalur_berhasil_pct, color="#e5e9ef", label="jalur berhasil")
ax.barh(models, final.jalur_optimal_pct, color=colors, label="jalur optimal")
ax.set_xlabel("% pasangan evaluasi"); ax.set_title("Hasil akhir: berhasil (abu) vs optimal (warna)", fontsize=10); ax.invert_yaxis()
plt.tight_layout(); plt.savefig(OUT / "5_federated_kurva.png", dpi=150); plt.show()

def show_round(ronde=RONDE):
    p = round_paths[ronde - 1]
    fig, ax = plt.subplots(figsize=(6.5, 5))
    draw_base(ax, node_size=170)
    nx.draw_networkx_edges(G, POS, ax=ax, edgelist=path_edges(oracle_path), edge_color="#d6006f", width=8, alpha=0.3)
    if p:
        nx.draw_networkx_edges(G, POS, ax=ax, edgelist=path_edges(p), edge_color="black", width=2.5, style="dashed")
    nx.draw_networkx_nodes(G, POS, ax=ax, nodelist=[SRC, DST], node_color="black", node_size=220)
    status = f"{path_metrics(p)['delay_aktual_ms']:.1f} ms (oracle {oracle_len[SRC][DST]:.1f} ms)" if p else "belum ada jalur"
    ax.set_title(f"Jalur model global {SRC}→{DST}, ronde {ronde}: {status}", fontsize=10); plt.show()

if HAS_WIDGETS:
    interact(show_round, ronde=widgets.IntSlider(min=1, max=RONDE, value=RONDE, continuous_update=False))
else:
    for rr in sorted({1, max(1, RONDE // 2), RONDE}):
        show_round(rr)

## 6. Traffic engineering: beberapa flow besar sekaligus
Beberapa flow (mis. backup/streaming) berlaju `LAJU_FLOW_MBPS` dikirim bersamaan **di atas trafik latar** (utilisasi bagian 2). Kapasitas sisa link = bandwidth × (1 − ρ). Strategi:
- **Hop-Count** dan **Dijkstra (delay dasar)** – tidak memperhatikan beban;
- **Load-Aware** – untuk tiap flow memilih jalur dari *k*-shortest-path yang meminimalkan beban maksimum link (kapasitas sisa).

Throughput tiap flow dihitung dengan alokasi **max-min fair** (progressive filling).

In [ ]:
#@title ⚙️ Parameter traffic engineering
JUMLAH_FLOW = 4  #@param {type:"slider", min:1, max:12, step:1}
LAJU_FLOW_MBPS = 15  #@param {type:"slider", min:1, max:100, step:1}
K_JALUR_KANDIDAT = 5  #@param {type:"slider", min:2, max:10, step:1}

In [ ]:
#@title 🚚 Jalankan skenario load balancing
RESID = {}
for u, v, d in G.edges(data=True):
    RESID[(u, v)] = RESID[(v, u)] = max(0.5, d["bandwidth"] * (1 - d["util"]))

used, FLOWS = set(), []
for s, t in sorted(pairs, key=lambda p: -hop_all[p[0]][p[1]]):
    if s not in used and t not in used:
        FLOWS.append((s, t)); used |= {s, t}
    if len(FLOWS) == JUMLAH_FLOW:
        break

def assign(strategy):
    load, out = defaultdict(float), {}
    for s, t in FLOWS:
        if strategy == "Hop-Count":
            p = nx.shortest_path(G, s, t)
        elif strategy == "Dijkstra (delay dasar)":
            p = nx.dijkstra_path(G, s, t, weight="base_delay")
        else:
            cands = list(itertools.islice(nx.shortest_simple_paths(G, s, t, weight="delay"), K_JALUR_KANDIDAT))
            p = min(cands, key=lambda c: (max((load[e] + LAJU_FLOW_MBPS) / RESID[e] for e in path_edges(c)),
                                          sum(G.edges[e]["delay"] for e in path_edges(c))))
        for e in path_edges(p):
            load[e] += LAJU_FLOW_MBPS
        out[(s, t)] = p
    return out, load

def maxmin_fair(paths):
    rate = {f: 0.0 for f in paths}; rem = dict(RESID); active = set(paths)
    while active:
        users = defaultdict(list)
        for f in active:
            for e in path_edges(paths[f]):
                users[e].append(f)
        inc = min([rem[e] / len(fs) for e, fs in users.items()] + [LAJU_FLOW_MBPS - rate[f] for f in active])
        for f in active:
            rate[f] += inc
        for e, fs in users.items():
            rem[e] -= inc * len(fs)
        done = {f for e, fs in users.items() if rem[e] <= 1e-9 for f in fs} | {f for f in active if LAJU_FLOW_MBPS - rate[f] <= 1e-9}
        active -= done
    return rate

STRATS = ["Hop-Count", "Dijkstra (delay dasar)", "Load-Aware"]
te_rows, te_load = [], {}
for st in STRATS:
    paths_st, load = assign(st)
    rate = maxmin_fair(paths_st)
    rates = np.array([rate[f] for f in FLOWS])
    te_load[st] = load
    te_rows.append(dict(strategi=st, throughput_total_mbps=rates.sum(), demand_total_mbps=LAJU_FLOW_MBPS * len(FLOWS),
                        flow_terendah_mbps=rates.min(), beban_maks_link=max(load[e] / RESID[e] for e in load),
                        jalur=" | ".join("→".join(p) for p in paths_st.values())))
te = pd.DataFrame(te_rows).set_index("strategi")
te.to_csv(OUT / "6_traffic_engineering.csv")
print("Flow:", FLOWS)
display(te.round(2))

fig, axes = plt.subplots(1, len(STRATS) + 1, figsize=(5 * (len(STRATS) + 1), 4.3))
for ax, st in zip(axes, STRATS):
    val = [max(te_load[st].get((a, b), 0), te_load[st].get((b, a), 0)) / RESID[(a, b)] for a, b in G.edges()]
    nx.draw_networkx_edges(G, POS, ax=ax, edge_color=np.clip(val, 0, 1.5), edge_cmap=plt.cm.RdYlGn_r, edge_vmin=0, edge_vmax=1.5,
                           width=[1 + 4 * min(v, 1.5) for v in val])
    draw_base(ax, width=0, node_size=140, labels=False)
    ax.set_title(f"{st}\nbeban maks {te.loc[st, 'beban_maks_link']:.2f} · {te.loc[st, 'throughput_total_mbps']:.0f}/{LAJU_FLOW_MBPS * len(FLOWS)} Mbps", fontsize=9)
ax = axes[-1]
ax.bar(STRATS, te.throughput_total_mbps, color=["#9aa5b1", "#2f6db5", "#1baf7a"])
ax.axhline(LAJU_FLOW_MBPS * len(FLOWS), color="black", ls="--", lw=1, label="total demand")
ax.set_ylabel("throughput diterima (Mbps)"); ax.tick_params(axis="x", rotation=15, labelsize=8); ax.legend(fontsize=8)
plt.suptitle("Beban flow / kapasitas sisa per link (merah = melebihi kapasitas)")
plt.tight_layout(); plt.savefig(OUT / "6_traffic_engineering.png", dpi=150); plt.show()

## 7. Ringkasan & unduh hasil

In [ ]:
#@title 💾 Simpan & unduh semua hasil
UNDUH_ZIP = False  #@param {type:"boolean"}
H = G.copy()
for u in H:
    x, y = H.nodes[u].pop("pos"); H.nodes[u]["x"], H.nodes[u]["y"] = float(x), float(y)
for u, v, d in H.edges(data=True):
    for k in list(d):
        d[k] = float(d[k]) if isinstance(d[k], (int, float, np.floating)) else str(d[k])
nx.write_graphml(H, OUT / "topologi.graphml")
links_df = pd.DataFrame([dict(source=u, target=v, **d) for u, v, d in G.edges(data=True)])
links_df.to_csv(OUT / "links.csv", index=False)
pd.DataFrame([dict(node=u, x=POS[u][0], y=POS[u][1], region=G.nodes[u]["region"]) for u in NODES]).to_csv(OUT / "nodes.csv", index=False)
for f in sorted(OUT.iterdir()):
    print(f"  {f}  ({f.stat().st_size / 1024:.1f} KB)")
if UNDUH_ZIP:
    zip_path = shutil.make_archive("hasil_networkx_01", "zip", OUT)
    if IN_COLAB:
        from google.colab import files
        files.download(zip_path)
    else:
        print("ZIP:", zip_path)

### Ide eksperimen
1. Naikkan `SKALA_TRAFIK` ke 1.4–1.6: selisih *Dijkstra (delay dasar)* terhadap oracle melebar — rute statis gagal saat jaringan padat.
2. Ubah `JAM_SIMULASI`: region dengan jam puncak berbeda membuat jalur optimal berpindah sepanjang hari.
3. RL: set `PENALTI_LOOP = 0` atau `BOBOT_KEMAJUAN = 0` dan amati agen lebih lama konvergen; naikkan `PENALTI_KONGESTI` agar agen menghindari link padat.
4. Federated: set `LOKALITAS_TUJUAN = 1.0` (klien hanya melihat tujuan lokal) → model **lokal** gagal untuk pasangan antar-region, sedangkan **global** tetap bisa karena tabel-Q digabung.
5. Bandingkan `METODE_AGREGASI` `visit_weighted` vs `fedavg` pada data yang sangat non-IID.